# Bengaluru Housing Analytics & Price Prediction

A structured datathon notebook with professional EDA, feature engineering, model comparison and production-ready prediction flow.

## 1. Setup

Import the libraries and reusable modules from `src/`. Keep analysis, not production code, in the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TRAIN_PATH, TEST_PATH, OUTPUTS_DIR
from src.preprocessing import basic_clean, remove_training_outliers
from src.feature_engineering import add_real_estate_features, add_train_based_location_features, reduce_rare_categories
from src.modeling import evaluate_models, fit_model, predict_prices, split_features_target
from src.utils import detect_column, summarize_dataframe

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)


## 2. Load data

Load the train dataset and check whether an optional test dataset is available.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None
train.shape, None if test is None else test.shape

## 3. Data overview

Inspect columns, missing values, and sample rows before making changes.

In [ ]:
print("Columns:", list(train.columns))
print("
Data types:
", train.dtypes)
print("
Missing values:
", train.isna().sum())
train.head(5)

## 4. Target and price distribution

Confirm the prediction target, then compare the raw and log-transformed distributions to validate RMSLE modeling.

In [ ]:
target_col = detect_column(train.columns, ["price", "Price", "target", "SalePrice"])
print("Target column:", target_col)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train[target_col], kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Raw price distribution")
sns.histplot(np.log1p(train[target_col]), kde=True, ax=axes[1], color="#ff7f0e")
axes[1].set_title("Log-transformed price distribution")
plt.tight_layout()

## 5. Cleaning and feature engineering

Use the reusable pipeline to clean location names, derive area-based metrics, and flag premium / efficiency signals.

In [ ]:
train_fe = basic_clean(train)
train_fe = remove_training_outliers(train_fe, target_col)
train_fe = add_real_estate_features(train_fe)

if test is not None:
    test_fe = basic_clean(test)
    test_fe = add_real_estate_features(test_fe)
    train_fe, test_fe = add_train_based_location_features(train_fe, test_fe, target_col)
    train_fe, test_fe = reduce_rare_categories(train_fe, test_fe, min_count=10)
else:
    test_fe = None

print("Cleaned shape:", train_fe.shape)
train_fe[["location_clean", "total_sqft_clean", "sqft_per_bhk", "bath_per_bhk", "is_furnished", "is_premium_location_keyword"]].head()

## 6. Key feature distributions

Review the most important engineered variables and understand how they relate to price.

In [ ]:
if {"total_sqft_clean", target_col}.issubset(train_fe.columns):
    train_fe["price_per_sqft"] = train_fe[target_col] / train_fe["total_sqft_clean"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(train_fe["price_per_sqft"].dropna(), kde=True, ax=axes[0], color="#2ca02c")
    axes[0].set_title("Price per sqft distribution")
    sns.histplot(train_fe["sqft_per_bhk"].dropna(), kde=True, ax=axes[1], color="#d62728")
    axes[1].set_title("Sqft per BHK distribution")
    plt.tight_layout()

## 7. Locality-level insights

Compare the top localities by count and by median price per sqft. This helps justify location-based features.

In [ ]:
if {"location_clean", "price_per_sqft"}.issubset(train_fe.columns):
    top_locations = train_fe["location_clean"].value_counts().head(12).index
    rank_df = train_fe[train_fe["location_clean"].isin(top_locations)].groupby("location_clean")["price_per_sqft"].median().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    rank_df.plot(kind="barh", color="#9467bd", ax=ax)
    ax.set_xlabel("Median price per sqft")
    ax.set_ylabel("Location")
    ax.set_title("Top locations by median price per sqft")
    plt.tight_layout()

## 8. Model comparison and selection

Evaluate baseline models using log-price RMSLE and choose the best performer for production.

In [ ]:
X, y = split_features_target(train_fe, target_col)
scores = evaluate_models(X, y, n_splits=5)
scores.style.format({"mean_rmsle": "{:.4f}", "std_rmsle": "{:.4f}"})

## 9. Train final model

Train the selected model and save the best model for the web prediction app.

In [ ]:
best_model_name = scores.loc[scores["mean_rmsle"].idxmin(), "model"]
print("Best model:", best_model_name)
model = fit_model(X, y, model_name=best_model_name)
import joblib
joblib.dump(model, Path("models") / "best_model.joblib")
print("Saved best model to models/best_model.joblib")

## 10. Next steps

- Add interactive locality explorer and map in the dashboard.
- Add SHAP explainability for individual predictions.
- Prepare an investor-friendly summary slide deck.
- Extend the UI to show affordability and investment score.